In [ ]:
!pip install -U transformers accelerate peft datasets bitsandbytes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 14.6 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [ ]:
from huggingface_hub import login
login(token="hf_token")


In [ ]:
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model


In [ ]:
dataset = load_dataset("zou-lab/MedCaseReasoning")
train_ds = dataset["train"]
val_ds   = dataset["val"]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/897 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/99.3M [00:00<?, ?B/s]

data/val-00000-of-00001.parquet:   0%|          | 0.00/3.78M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/7.35M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/13092 [00:00<?, ? examples/s]

Generating val split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/897 [00:00<?, ? examples/s]

In [ ]:
model_name = "Qwen/Qwen3-1.7B"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    use_fast=True
)

# Qwen does not always define pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

model.config.use_cache = False


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.44G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/622M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [ ]:
MAX_LEN = 768

# ===============================
# Reasoning Stitching (Paper Technique)
# ===============================
def stitch_reasoning(case, reasoning, diagnosis):
    """
    Converts enumerated clinician reasoning into a single
    coherent reasoning trace without adding new information.
    """
    return (
        "Clinical Case:\n"
        f"{case}\n\n"
        "Diagnostic Reasoning:\n"
        "The clinician considers the following diagnostic possibilities and evidence. "
        "Each alternative diagnosis is evaluated and excluded or supported based on "
        "the clinical findings provided.\n\n"
        f"{reasoning}\n\n"
        "Final Diagnosis:\n"
        f"{diagnosis}"
    )


In [ ]:
def preprocess(batch):
    input_ids_list = []
    attention_masks = []
    labels_list = []

    for case, reasoning, diagnosis in zip(
        batch["case_prompt"],
        batch["diagnostic_reasoning"],
        batch["final_diagnosis"]
    ):
        # Prompt shown to the model (no loss)
        prompt = (
            "You are a medical expert.\n\n"
            "Read the following clinical case and provide a detailed diagnostic "
            "reasoning process, followed by the final diagnosis.\n\n"
        )

        # Paper-style stitched reasoning + diagnosis (loss applied)
        answer = stitch_reasoning(case, reasoning, diagnosis)

        full_text = prompt + answer

        tokenized = tokenizer(
            full_text,
            truncation=True,
            max_length=MAX_LEN,
            padding="max_length",
        )

        input_ids = tokenized["input_ids"]
        attention_mask = tokenized["attention_mask"]

        # Mask prompt tokens (no loss on prompt)
        prompt_ids = tokenizer(
            prompt,
            add_special_tokens=False
        )["input_ids"]

        labels = [-100] * len(prompt_ids) + input_ids[len(prompt_ids):]
        labels = labels[:MAX_LEN]
        labels += [-100] * (MAX_LEN - len(labels))

        input_ids_list.append(input_ids)
        attention_masks.append(attention_mask)
        labels_list.append(labels)

    return {
        "input_ids": input_ids_list,
        "attention_mask": attention_masks,
        "labels": labels_list,
    }

In [ ]:
train_ds = train_ds.map(
    preprocess,
    batched=True,
    remove_columns=train_ds.column_names,
    load_from_cache_file=False,
)

val_ds = val_ds.map(
    preprocess,
    batched=True,
    remove_columns=val_ds.column_names,
    load_from_cache_file=False,
)


Map:   0%|          | 0/13092 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 3,211,264 || all params: 1,723,786,240 || trainable%: 0.1863


In [ ]:
training_args = TrainingArguments(
    output_dir="./qwen3_medcase",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,  # effective batch size = 32
    learning_rate=2e-5,
    num_train_epochs=1,  # paper uses 3; 1 is reasonable for 1.7B
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    fp16=True,
    report_to="none",
    save_total_limit=2,
    load_best_model_at_end=True,
)


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
)


/tmp/ipython-input-3150992467.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
trainer.train()


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Epoch,Training Loss,Validation Loss


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

gdrive_dir = "/content/drive/MyDrive/llm_checkpoints/qwen3"
os.makedirs(gdrive_dir, exist_ok=True)

trainer.model.save_pretrained(gdrive_dir)
tokenizer.save_pretrained(gdrive_dir)


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
# ===============================
# Install dependencies
# ===============================
!pip install -U transformers accelerate peft datasets bitsandbytes

# ===============================
# Imports
# ===============================
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model

# ===============================
# Dataset
# ===============================
dataset = load_dataset("zou-lab/MedCaseReasoning")
train_ds = dataset["train"]
val_ds   = dataset["val"]

# ===============================
# Model & Tokenizer
# ===============================
BASE_MODEL = "Qwen/Qwen3-1.7B"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)

# Add explicit END token (CRITICAL FIX)
special_tokens = {"additional_special_tokens": ["<END>"]}
tokenizer.add_special_tokens(special_tokens)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto"
)

model.resize_token_embeddings(len(tokenizer))
model.config.use_cache = False

MAX_LEN = 768

# ===============================
# Reasoning Stitching
# ===============================
def stitch_reasoning(case, reasoning, diagnosis):
    """
    Convert enumerated clinician reasoning into a
    single coherent diagnostic reasoning trace.
    """
    return (
        "Diagnostic Reasoning:\n"
        f"{reasoning}\n\n"
        "Final Diagnosis:\n"
        f"{diagnosis}\n"
        "<END>"
    )

# ===============================
# Preprocessing (SFT with Masking)
# ===============================
def preprocess(batch):
    input_ids_list = []
    attention_masks = []
    labels_list = []

    for case, reasoning, diagnosis in zip(
        batch["case_prompt"],
        batch["diagnostic_reasoning"],
        batch["final_diagnosis"]
    ):
        prompt = (
            "You are a medical expert.\n\n"
            "Given the following clinical case, explain the diagnostic reasoning "
            "step by step, then provide exactly one final diagnosis.\n\n"
            "Clinical Case:\n"
            f"{case}\n\n"
        )

        target = stitch_reasoning(case, reasoning, diagnosis)

        full_text = prompt + target

        enc = tokenizer(
            full_text,
            truncation=True,
            max_length=MAX_LEN,
            padding="max_length",
        )

        input_ids = enc["input_ids"]
        attention_mask = enc["attention_mask"]

        # Mask prompt tokens
        prompt_len = len(
            tokenizer(prompt, add_special_tokens=False)["input_ids"]
        )

        labels = [-100] * prompt_len + input_ids[prompt_len:]
        labels = labels[:MAX_LEN]
        labels += [-100] * (MAX_LEN - len(labels))

        input_ids_list.append(input_ids)
        attention_masks.append(attention_mask)
        labels_list.append(labels)

    return {
        "input_ids": input_ids_list,
        "attention_mask": attention_masks,
        "labels": labels_list,
    }

# ===============================
# Apply preprocessing
# ===============================
train_ds = train_ds.map(
    preprocess,
    batched=True,
    remove_columns=train_ds.column_names,
    load_from_cache_file=False,
)

val_ds = val_ds.map(
    preprocess,
    batched=True,
    remove_columns=val_ds.column_names,
    load_from_cache_file=False,
)

# ===============================
# LoRA (Practical SFT)
# ===============================
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# ===============================
# Training Arguments
# ===============================
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/llm_checkpoints/qwen3_medcase_improved",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    num_train_epochs=1,          # improved convergence
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    fp16=True,
    report_to="none",
    save_total_limit=2,
    load_best_model_at_end=True,
)

# ===============================
# Trainer
# ===============================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
)

# ===============================
# Train
# ===============================
trainer.train()

# ===============================
# Save
# ===============================
trainer.model.save_pretrained("/content/drive/MyDrive/llm_checkpoints/qwen3_medcase_lora")
tokenizer.save_pretrained("/content/drive/MyDrive/llm_checkpoints/qwen3_medcase_lora")


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# Paths
MODEL_DIR = "/content/drive/MyDrive/llm_checkpoints/qwen3_medcase_lora"
BASE_MODEL = "Qwen/Qwen3-1.7B"

# 1. Load tokenizer FROM THE ADAPTER DIR (includes <END>)
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 2. Load base model
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
)

# 3. 🔴 CRITICAL: resize embeddings BEFORE loading LoRA
base_model.resize_token_embeddings(len(tokenizer))

# 4. Load LoRA adapter
model = PeftModel.from_pretrained(base_model, MODEL_DIR)
model.eval()



In [ ]:
def build_prompt(case_prompt: str) -> str:
    return (
        "You are a medical expert.\n\n"
        "Given the following clinical case, explain the diagnostic reasoning "
        "step by step, then provide exactly one final diagnosis.\n\n"
        "Clinical Case:\n"
        f"{case_prompt}\n\n"
    )


In [ ]:
def run_inference(
    case_prompt: str,
    max_new_tokens: int = 256,
):
    prompt = build_prompt(case_prompt)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        padding=True,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,                 # CRITICAL
            repetition_penalty=1.2,
            eos_token_id=tokenizer.convert_tokens_to_ids("<END>"),
        )

    decoded = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    # Remove prompt from output
    answer = decoded[len(prompt):].strip()

    # Safety trim if END appears in text
    if "<END>" in answer:
        answer = answer.split("<END>")[0].strip()

    return answer


In [ ]:
case = """
A 38-year-old woman presents with progressive shortness of breath and dry cough
over several months. Chest imaging shows bilateral hilar lymphadenopathy.
Laboratory studies reveal elevated serum ACE levels. There is no fever, weight
loss, or history of tuberculosis exposure.

"""

response = run_inference(case)
print(response)
